# ccdp — damage-seg Path A training (CarDD + HITL filtered, nc=6)

**Path A:** keep the existing nc=6 head and warm-start from the current `damage_seg.pt`.
Adds HITL examples for the **overlapping classes only** — no head re-init, no big LR
warmup, just more data for the same labels.

CarDD's 6 classes (you must keep this exact order):
```
0 dent  1 scratch  2 crack  3 glass-shatter  4 lamp-broken  5 tire-flat
```

HITL ships 8 damage classes; we map each one to a CarDD index (or `None` to drop it).
You fill in the dict in section 4. Anything mapped to `None` is excluded from training.

### Works on both Colab and Kaggle
- **Colab:** mounts Drive, caches the dataset there, checkpoints to Drive.
- **Kaggle:** uses `RCLONE_CONF` Secret + GDrive for the same purpose.
Cell 1 auto-detects the platform.

### Runtime
~30–40 min/epoch on T4 with combined ~5.8k images, batch 16, imgsz 640.
Path A target: 30–40 epochs warm-start → ~1.5–2 h. Free-tier safe.


## 1. Platform autodetect + GPU check

In [ ]:
import os, sys, pathlib
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
ON_KAGGLE = os.path.exists('/kaggle')
PLATFORM  = 'colab' if ON_COLAB else 'kaggle' if ON_KAGGLE else 'local'
print('platform:', PLATFORM)

import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'enable GPU')

## 2. Mount durable storage

On Colab → mount `MyDrive`. On Kaggle → install rclone from the `RCLONE_CONF` secret
and use `gdrive:` as a remote. Either way we end up with a `DURABLE_ROOT` path
that survives a kernel crash.

In [ ]:
import os, subprocess, pathlib

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DURABLE_ROOT = '/content/drive/MyDrive/ccdp'
    DURABLE_IS_FS = True   # we can write straight to it
elif PLATFORM == 'kaggle':
    DURABLE_ROOT = '/kaggle/working/_drive'   # local staging; rclone mirrors to gdrive
    GDRIVE_REMOTE = 'gdrive:ccdp'
    DURABLE_IS_FS = False
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret('RCLONE_CONF')
        subprocess.run('which rclone || (curl -sL https://rclone.org/install.sh | sudo bash)',
                       shell=True, check=False)
        cfg_dir = pathlib.Path.home() / '.config' / 'rclone'
        cfg_dir.mkdir(parents=True, exist_ok=True)
        (cfg_dir / 'rclone.conf').write_text(conf)
        print('[rclone] configured')
    except Exception as e:
        print(f'[rclone] skipping sync — {e!r}')
        GDRIVE_REMOTE = None
else:
    DURABLE_ROOT = str(pathlib.Path.home() / 'ccdp-durable')
    DURABLE_IS_FS = True

os.makedirs(DURABLE_ROOT, exist_ok=True)
CKPT_ROOT_DURABLE = f'{DURABLE_ROOT}/checkpoints/segmenter'
os.makedirs(CKPT_ROOT_DURABLE, exist_ok=True)
print('durable root:', DURABLE_ROOT)

## 3. Clone repo + install

In [ ]:
WORK = '/content' if PLATFORM == 'colab' else '/kaggle/working' if PLATFORM == 'kaggle' else os.path.expanduser('~')
os.chdir(WORK)
!rm -rf car-crash-fix-amount-predictor
!git clone --depth 1 https://github.com/theDocWho/car-crash-fix-amount-predictor.git
os.chdir(f'{WORK}/car-crash-fix-amount-predictor')
!pip -q install -e '.[ml]' ultralytics

# Symlink checkpoints/segmenter -> durable storage so saves persist.
import pathlib
pathlib.Path('checkpoints').mkdir(exist_ok=True)
link = pathlib.Path('checkpoints/segmenter')
if link.exists() or link.is_symlink(): link.unlink()
if DURABLE_IS_FS:
    link.symlink_to(CKPT_ROOT_DURABLE)
else:
    link.mkdir()                   # local; we'll rclone-mirror periodically
    os.makedirs(CKPT_ROOT_DURABLE, exist_ok=True)
print('checkpoints/segmenter ->', os.readlink(link) if link.is_symlink() else '(local + rclone mirror)')

In [ ]:
# Defensive sys.path nudge — on Colab, `pip install -e .` registers the package
# but the active kernel sometimes doesn't pick up the new entry until the next
# cell. Without this, §5's `from ccdp.data import cardd_yolo` raises ImportError
# on a fresh runtime. Re-runs are harmless.
import sys, os
# WORK was set in cell 6 — re-derive defensively in case the cell was re-run.
_work = '/content' if PLATFORM == 'colab' else '/kaggle/working' if PLATFORM == 'kaggle' else os.path.expanduser('~')
_src = os.path.join(_work, 'car-crash-fix-amount-predictor', 'src')
if _src not in sys.path:
    sys.path.append(_src)

import ccdp
print('ccdp package successfully installed!')
print('Location:', ccdp.__file__)


## 4. **EDIT ME** — HITL → CarDD class remap

Fill in this dict before training. HITL class names are in the dataset's README — set
the value to the matching CarDD index `0..5`, or `None` to drop that class from
training. Anything not listed here is also dropped.

CarDD ordering:
```
0 dent  1 scratch  2 crack  3 glass-shatter  4 lamp-broken  5 tire-flat
```

Below is a starter guess based on common HITL ontologies — **verify against the
actual HITL class names before kicking off a long run.**

In [ ]:
HITL_TO_CARDD = {
    # HITL class name (lowercase, exact)       -> CarDD index 0..5 or None
    'dent':            0,
    'scratch':         1,
    'crack':           2,
    'shattered_glass': 3,
    'broken_lamp':     4,
    'flat_tire':       5,
    # HITL classes with no CarDD equivalent — drop:
    'paint_chip':   None,
    'rust':         None,
}
# Sanity: number of mapped-in classes
mapped = sum(1 for v in HITL_TO_CARDD.values() if v is not None)
print(f'HITL classes kept: {mapped}/{len(HITL_TO_CARDD)} (rest dropped)')

## 5. Fetch CarDD + HITL

CarDD comes from the repo's data prep. HITL pulls from the Kaggle CC0 mirror
(replace the slug if you use a different one).

In [ ]:
import os, json, pathlib, glob, getpass

HITL_SLUG = 'YOUR_USERNAME/hitl-vehicle-damage'   # <-- update with actual Kaggle slug
HITL_DRIVE_CACHE = f'{DURABLE_ROOT}/data/raw/hitl'
os.makedirs(HITL_DRIVE_CACHE, exist_ok=True)

# --- Kaggle creds ------------------------------------------------------
# New flow: KAGGLE_USERNAME + KAGGLE_KEY env vars (token from Kaggle → Settings → API).
# Fallbacks: Drive token file -> legacy kaggle.json -> interactive prompt.
def _resolve_kaggle_creds():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return 'env'
    if PLATFORM == 'kaggle':
        # Kaggle kernels are auto-authenticated for their own datasets — nothing to do.
        return 'kaggle-native'
    # Colab: try the durable token file, then legacy json, then prompt.
    token_file = '/content/drive/MyDrive/ccdp/kaggle_token.txt'
    legacy_json = '/content/drive/MyDrive/kaggle/kaggle.json'
    if os.path.exists(token_file):
        for line in pathlib.Path(token_file).read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line: continue
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")
        if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
            return f'drive token file ({token_file})'
    if os.path.exists(legacy_json):
        data = json.loads(pathlib.Path(legacy_json).read_text())
        os.environ['KAGGLE_USERNAME'] = data['username']
        os.environ['KAGGLE_KEY'] = data['key']
        return f'legacy kaggle.json ({legacy_json})'
    print('No Kaggle creds found. Paste them from Kaggle → Settings → API.')
    os.environ['KAGGLE_USERNAME'] = input('KAGGLE_USERNAME: ').strip()
    os.environ['KAGGLE_KEY'] = getpass.getpass('KAGGLE_KEY (token): ').strip()
    if input(f'Save to {token_file} for next session? [y/N] ').strip().lower() == 'y':
        pathlib.Path(token_file).parent.mkdir(parents=True, exist_ok=True)
        pathlib.Path(token_file).write_text(
            f"KAGGLE_USERNAME={os.environ['KAGGLE_USERNAME']}\n"
            f"KAGGLE_KEY={os.environ['KAGGLE_KEY']}\n"
        )
        os.chmod(token_file, 0o600)
        print(f'saved → {token_file}')
    return 'interactive'

source = _resolve_kaggle_creds()
print(f'[kaggle] auth via {source} (user={os.environ.get("KAGGLE_USERNAME","<kaggle-native>")})')

# CarDD — uses our packaged builder
from ccdp.data import cardd_yolo
cardd_yaml = cardd_yolo.build()
print('CarDD yaml:', cardd_yaml)

# HITL — download if not cached on Drive
if not glob.glob(f'{HITL_DRIVE_CACHE}/*'):
    !pip -q install kaggle
    !kaggle datasets download -d {HITL_SLUG} -p {HITL_DRIVE_CACHE} --unzip
else:
    print('HITL cache present:', HITL_DRIVE_CACHE)

## 6. Build the combined YOLO dataset (CarDD + remapped HITL)

We *don't* touch CarDD's labels. For HITL we rewrite each `.txt` label so the class id
column is the CarDD index from `HITL_TO_CARDD` — and skip any label line whose original
class maps to `None`. Skipped images with no remaining boxes are dropped from the manifest.

In [ ]:
import shutil, pathlib, yaml, re
from pathlib import Path

OUT_ROOT = Path('data/processed/cardd_hitl_pathA')
for split in ('train', 'val'):
    (OUT_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

# --- Step A: link CarDD images + labels into combined tree (no changes)
import yaml
cardd_cfg = yaml.safe_load(open(cardd_yaml))
cardd_root = Path(cardd_cfg['path']) if 'path' in cardd_cfg else Path(cardd_yaml).parent
for split in ('train', 'val'):
    img_dir = cardd_root / cardd_cfg[split] if isinstance(cardd_cfg[split], str) else None
    if not img_dir or not img_dir.exists():
        # CarDD yaml may use direct paths; fall back to canonical layout
        img_dir = cardd_root / 'images' / split
    lbl_dir = cardd_root / 'labels' / split
    for p in img_dir.glob('*'):
        tgt = OUT_ROOT/'images'/split/p.name
        if not tgt.exists(): tgt.symlink_to(p.resolve())
    for p in lbl_dir.glob('*.txt'):
        tgt = OUT_ROOT/'labels'/split/p.name
        if not tgt.exists(): tgt.symlink_to(p.resolve())

# --- Step B: walk HITL, remap classes, write to combined tree
# Assumes HITL ships YOLO-format labels with a `classes.txt` mapping row-id -> name.
hitl_root = Path(HITL_DRIVE_CACHE)
classes_txt = next(hitl_root.rglob('classes.txt'), None)
assert classes_txt, f'no classes.txt in {hitl_root} — adjust this cell to match your HITL layout'
hitl_id_to_name = {i: n.strip().lower() for i, n in enumerate(classes_txt.read_text().splitlines()) if n.strip()}
print('HITL classes detected:', hitl_id_to_name)

# Build the remap: old_id -> new CarDD id (or None to drop)
hitl_remap = {old_id: HITL_TO_CARDD.get(name) for old_id, name in hitl_id_to_name.items()}
print('remap:', hitl_remap)

import random
random.seed(42)
kept, dropped = 0, 0
for lbl in hitl_root.rglob('*.txt'):
    if lbl.name == 'classes.txt': continue
    new_lines = []
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if not parts: continue
        old = int(parts[0])
        new = hitl_remap.get(old)
        if new is None: continue
        new_lines.append(' '.join([str(new), *parts[1:]]))
    if not new_lines:
        dropped += 1; continue
    img_candidates = list(lbl.parent.glob(lbl.stem + '.*'))
    img_candidates = [p for p in img_candidates if p.suffix.lower() in {'.jpg','.jpeg','.png'}]
    if not img_candidates: continue
    img = img_candidates[0]
    split = 'val' if random.random() < 0.1 else 'train'
    (OUT_ROOT/'labels'/split/f'hitl_{lbl.stem}.txt').write_text('\n'.join(new_lines))
    tgt_img = OUT_ROOT/'images'/split/f'hitl_{img.name}'
    if not tgt_img.exists(): tgt_img.symlink_to(img.resolve())
    kept += 1

print(f'HITL kept: {kept}  dropped (no overlapping classes): {dropped}')

# Write combined data.yaml (nc=6 — Path A)
combined_yaml = OUT_ROOT / 'data.yaml'
combined_yaml.write_text(yaml.safe_dump({
    'path': str(OUT_ROOT.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'nc': 6,
    'names': ['dent','scratch','crack','glass-shatter','lamp-broken','tire-flat'],
}))
print('combined data.yaml:', combined_yaml)

## 7. Fetch the existing damage_seg base weights (warm-start source)

In [ ]:
import os
BASE_SEG = 'checkpoints/production/damage_seg.pt'
os.makedirs('checkpoints/production', exist_ok=True)
if not os.path.exists(BASE_SEG):
    !curl -fL --retry 3 -o {BASE_SEG} \
       https://github.com/theDocWho/car-crash-fix-amount-predictor/releases/download/v0.2.0/damage_seg.pt
print('base seg:', os.path.exists(BASE_SEG))

## 8. Train (warm-start, auto-resume via Ultralytics)

Ultralytics has a native `resume=True` that reads its `args.yaml` + `last.pt` and
continues. We bypass `ccdp train yolov8` for that and call `YOLO.train` directly so
we can flip resume on cleanly when a prior run dir is present.

Production values (substitute when you've sanity-checked):
- epochs = 40, batch = 16, imgsz = 640, patience = 15


In [ ]:
RUN_TRAINING = False   # flip to True to actually run

# Production values:
EPOCHS    = 40   # production
BATCH     = 16   # production
IMGSZ     = 640  # production
PATIENCE  = 15   # production

# Safety defaults so an accidental True doesn't burn a session:
SMOKE_EPOCHS, SMOKE_BATCH, SMOKE_IMGSZ = 1, 2, 320

if RUN_TRAINING:
    from ultralytics import YOLO
    import glob, pathlib
    prior = sorted(glob.glob('checkpoints/segmenter/run_*pathA*'))
    if prior and (pathlib.Path(prior[-1])/'ultralytics'/'weights'/'last.pt').exists():
        # Ultralytics-native resume: point YOLO at the run's last.pt
        last = pathlib.Path(prior[-1])/'ultralytics'/'weights'/'last.pt'
        print(f'[resume] continuing from {last}')
        model = YOLO(str(last))
        results = model.train(resume=True)
    else:
        # Fresh run: warm-start from production damage_seg.pt
        from ccdp.registry import create_run
        run_dir = create_run(variant='segmenter', tag='pathA_cardd_hitl_nc6',
                             notes='Path A: warm-start nc=6 with CarDD+HITL-remapped')
        print(f'[fresh] run dir: {run_dir}')
        model = YOLO(BASE_SEG)
        results = model.train(
            data=str(pathlib.Path('data/processed/cardd_hitl_pathA/data.yaml').resolve()),
            epochs=SMOKE_EPOCHS,   # production: EPOCHS
            batch=SMOKE_BATCH,     # production: BATCH
            imgsz=SMOKE_IMGSZ,     # production: IMGSZ
            patience=PATIENCE,
            project=str(run_dir.resolve()),
            name='ultralytics',
            exist_ok=True,
            plots=False, verbose=False,
        )
        # Bubble best.pt / last.pt to run_dir root (matches our registry layout)
        ult = run_dir/'ultralytics'/'weights'
        for fn in ('best.pt', 'last.pt'):
            src = ult/fn; dst = run_dir/fn
            if src.exists():
                if dst.is_symlink() or dst.exists(): dst.unlink()
                dst.symlink_to(src.relative_to(run_dir))
    print('[done]')
else:
    print('Skipped — flip RUN_TRAINING=True to execute.')

## 9. (Kaggle only) Background mirror to GDrive

On Colab the run dir lives on Drive already. On Kaggle we mirror `checkpoints/segmenter/`
to `gdrive:ccdp/checkpoints/segmenter/` every 60s.

In [ ]:
if PLATFORM == 'kaggle' and 'GDRIVE_REMOTE' in dir() and GDRIVE_REMOTE:
    import threading, time, glob, os, subprocess
    def _sync():
        while True:
            try:
                for d in glob.glob('checkpoints/segmenter/run_*'):
                    dest = f"{GDRIVE_REMOTE}/checkpoints/segmenter/{os.path.basename(d.rstrip('/'))}"
                    subprocess.run(['rclone','copy','--update','--quiet',d,dest], check=False)
            except Exception as e:
                print(f'[sync] {e!r}')
            time.sleep(60)
    threading.Thread(target=_sync, daemon=True).start()
    print('[gdrive] mirror running every 60s')
else:
    print('[gdrive] not applicable (Colab uses Drive directly)')

## 10. Export final weights

Copies the best.pt from the latest Path A run dir onto durable storage so you can
download + upload to the GitHub release.

In [ ]:
import glob, os, shutil
src = sorted(glob.glob('checkpoints/segmenter/run_*pathA*/best.pt'))
if src:
    real = os.path.realpath(src[-1])
    dst  = f'{DURABLE_ROOT}/damage_seg_pathA.pt'
    shutil.copy(real, dst)
    print(f'exported -> {dst}')
else:
    print('no best.pt — run training first')

## 11. Resume-from-crash recap

- **Colab:** reconnect runtime → re-run cells 1–8. Cell 2 remounts Drive, cell 3
  re-symlinks `checkpoints/segmenter`, cell 8 detects the prior `run_*pathA*` dir on
  Drive and calls `YOLO(last.pt).train(resume=True)` — Ultralytics resumes from the saved
  optimizer state at the next epoch.
- **Kaggle:** Save & Run All again. The rclone cell pulls the prior run dir back into
  `/kaggle/working/checkpoints/segmenter/`, then the same resume detection fires.

If you change `HITL_TO_CARDD` between launches, **delete the prior run dir first** —
Ultralytics' resume assumes the same data + nc; remapping classes mid-run silently
poisons the training.

### Push to release
```bash
cp /content/drive/MyDrive/ccdp/damage_seg_pathA.pt .
gh release upload v0.2.0 damage_seg.pt --clobber   # or a Path-A-specific asset name
```
